# DB2Model — QLoRA на Kaggle

Цель этого ноутбука — **проверить, что пайплайн вообще работает**, а не получить хороший результат.
Успех = модель загрузилась в 16 ГБ, обучение прошло, адаптер сохранился, предсказания выгрузились.

## Перед запуском

1. **Settings → Accelerator → GPU T4 x2** (или P100).
2. **Settings → Internet → On** (иначе модель не скачается с HuggingFace).
3. Залей как Kaggle Dataset папку с данными и подключи через **Add Input**:
   - `db2model/clean/toxicology.json` — обучающие пары
   - `data/bird_large.json` — вопросы для замера

   Путь появится как `/kaggle/input/<имя-датасета>/...` — поправь `DATA_DIR` в ячейке с настройками.

## Что забрать домой

- `/kaggle/working/adapter/` — обученный адаптер
- `/kaggle/working/query_results.json` — предсказания, их считать локально через `bird_evaluate_only.py`
  (в Kaggle нет доступа к базе через твой туннель, поэтому EX считается дома)

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

In [ ]:
import json, os, torch
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
DB_ID = "toxicology"

# ПОПРАВЬ ПОД СВОЙ ЗАЛИТЫЙ ДАТАСЕТ
DATA_DIR = Path("/kaggle/input/db2model-data")
TRAIN_FILE = DATA_DIR / f"{DB_ID}.json"
BIRD_FILE = DATA_DIR / "bird_large.json"

OUT_DIR = Path("/kaggle/working")
ADAPTER_DIR = OUT_DIR / "adapter"

print("GPU:", torch.cuda.get_device_name(0))
print("памяти всего:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "ГБ")
for f in (TRAIN_FILE, BIRD_FILE):
    print(("есть  " if f.exists() else "НЕТ   ") + str(f))

## Загрузка модели в 4 бита

Главная проверка: влезает ли 3B в память. Если тут упадёт по OOM — дальше идти незачем.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import transformers

print("transformers", transformers.__version__)

# T4 и P100 не умеют bfloat16 в железе, A100/L4 умеют. Тип обучения должен
# совпадать с типом весов, иначе GradScaler упадёт на несовпадении.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("считаем в", COMPUTE_DTYPE)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# transformers v5 переименовал torch_dtype в dtype. Под старым именем аргумент
# молча игнорируется, и модель грузится в своём родном bfloat16.
load_kwargs = dict(quantization_config=bnb, device_map={"": 0})
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=COMPUTE_DTYPE, **load_kwargs)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=COMPUTE_DTYPE, **load_kwargs)
model.config.use_cache = False

dtypes = {p.dtype for p in model.parameters()}
print("типы параметров:", dtypes)
assert COMPUTE_DTYPE in dtypes, f"веса не в {COMPUTE_DTYPE} — обучение упадёт на GradScaler"
print("занято на GPU:", round(torch.cuda.memory_allocated() / 1e9, 2), "ГБ")

In [ ]:
SYSTEM = f"You are a PostgreSQL expert for the database `{DB_ID}`. Return only SQL."

def build_prompt(question: str) -> str:
    """Схемы здесь нет намеренно: вся суть ветки в том, что знание о базе
    лежит в весах, а не в контексте."""
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True,
    )

def generate(question: str, max_new_tokens: int = 160) -> str:
    inputs = tokenizer(build_prompt(question), return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# До обучения модель про базу не знает — ответ будет выдуманным. Это нормально,
# это и есть точка отсчёта "zero-shot без схемы".
print(generate("How many molecules have a label of '+'?"))

## Данные

Пары приходят из `filter_pairs.py` — уже отфильтрованные исполнением на живой базе.

In [ ]:
from datasets import Dataset

pairs = json.loads(TRAIN_FILE.read_text(encoding="utf-8"))
print("пар в наборе:", len(pairs))

def to_text(p):
    return {"text": build_prompt(p["question"]) + p["sql"] + tokenizer.eos_token}

ds = Dataset.from_list([to_text(p) for p in pairs])
split = ds.train_test_split(test_size=0.15, seed=0)
print(split)
print("\nпример:\n", split["train"][0]["text"][:400])

## QLoRA

`r`, `alpha`, число эпох — то, что потом перебирается на неделе 4. Сейчас цель — просто доехать.

In [ ]:
from dataclasses import fields

import trl
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("trl", trl.__version__)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

use_bf16 = COMPUTE_DTYPE is torch.bfloat16
kwargs = dict(
    output_dir=str(OUT_DIR / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="epoch",
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    report_to="none",
)

# trl переименовал max_seq_length в max_length; какое имя живо — спрашиваем
# у самого класса, а не угадываем по номеру версии.
supported = {f.name for f in fields(SFTConfig)}
for name in ("max_length", "max_seq_length"):
    if name in supported:
        kwargs[name] = 512
        print("длина последовательности задана через", name)
        break
else:
    print("ВНИМАНИЕ: параметр длины не найден, остаётся умолчание")

dropped = set(kwargs) - supported
if dropped:
    print("эта версия trl не знает и я их не передаю:", dropped)
args = SFTConfig(**{k: v for k, v in kwargs.items() if k in supported})

trainer = SFTTrainer(
    model=model, args=args, train_dataset=split["train"],
    eval_dataset=split["test"], peft_config=peft_config,
)
trainer.train()
print("пик памяти:", round(torch.cuda.max_memory_allocated() / 1e9, 2), "ГБ")

In [ ]:
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("адаптер:", [f.name for f in ADAPTER_DIR.iterdir()])
print("размер, МБ:", round(sum(f.stat().st_size for f in ADAPTER_DIR.iterdir()) / 1e6, 1))

## Предсказания для замера

База отсюда недоступна, и она **не нужна**: модель получает только текст вопроса.
Файл забираешь домой и считаешь EX локально.

In [ ]:
import re

model.config.use_cache = True
trainer.model.eval()

questions = [q for q in json.loads(BIRD_FILE.read_text(encoding="utf-8"))
             if q["db_id"] == DB_ID]
print("вопросов по этой базе:", len(questions))

def clean_sql(text: str) -> str:
    text = re.sub(r"^```(?:sql)?|```$", "", text.strip(), flags=re.IGNORECASE)
    return text.strip().rstrip(";").strip()

predictions = {}
for i, q in enumerate(questions, 1):
    # evidence подаётся так же, как в baseline — иначе сравнение нечестное
    question = f"question: {q['question']}, evidence (may be empty): {q['evidence']}"
    predictions[str(q["question_id"])] = clean_sql(generate(question))
    if i % 10 == 0:
        print(f"  {i}/{len(questions)}")

out = OUT_DIR / "query_results.json"
out.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nготово:", out)
print(list(predictions.items())[0])

## Дома

Скачай `query_results.json`, подними туннель и посчитай EX:

```bash
PYTHONIOENCODING=utf-8 uv run --env-file .env python bird_evaluate_only.py \
    query_results.json data/bird_large.json
```

Число сравнивается с baseline (Qwen-7B со схемой в промпте, EX 22.73% на bird_small).
Осторожно: baseline снят на другой модели и другом наборе вопросов — для честного
сравнения baseline надо пересчитать на `bird_large` и на той же 3B.